# Kaza — Part 1: LiquidCodec Test (Twi & Igbo)

In [ ]:
!test -d /content/kaza && git -C /content/kaza pull --ff-only origin phase1-design || git clone -b phase1-design https://github.com/udohchuks/kaza.git /content/kaza
%cd /content/kaza
!pip install -q -r requirements.txt

In [ ]:
from kaza.env import print_env
from kaza.codec import KazaCodec

env = print_env()
codec = KazaCodec(device=env.device, dtype=env.frozen_dtype)

In [ ]:
import io, soundfile as sf
from datasets import Audio, load_dataset
from IPython.display import Audio as Play, display
from kaza.codec import SAMPLE_RATE, codec_metrics, to_mono_24k

def test_subset(name, config, n=3):
    ds = load_dataset("google/WaxalNLP", config, split="train", streaming=True)
    ds = ds.cast_column("audio", Audio(decode=False))
    print(f"\n=== {name} ({config}) ===")
    for i, row in enumerate(ds.take(n)):
        wav, sr = sf.read(io.BytesIO(row["audio"]["bytes"]))
        ref = to_mono_24k(wav, sr)
        tokens = codec.encode(ref)
        rec = codec.decode(tokens)[0]
        m = codec_metrics(ref, rec)
        print(f"[{i}] tokens={tuple(tokens.shape)} range=[{tokens.min().item()}, {tokens.max().item()}] | {m}")
        print("Original:"); display(Play(ref.numpy(), rate=SAMPLE_RATE))
        print("Reconstructed:"); display(Play(rec.numpy(), rate=SAMPLE_RATE))

test_subset("Twi (Akan)", "aka_asr", n=3)
test_subset("Igbo", "ibo_tts", n=1)